# Chapter 3 artifact checks

Notebook hỗ trợ tạo sơ đồ, phân bố pickup P95, bảng quy mô raw và audit uniqueness OD. Các lệnh này không chạy benchmark Spark Standalone; ND2/ND3 vẫn chạy qua các benchmark notebooks/runners riêng.

In [1]:
from pathlib import Path
import os, subprocess, sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / 'scripts').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Không tìm thấy project root có thư mục scripts')
os.chdir(ROOT)


## Sơ đồ kiến trúc và topology
Chạy lại script để tạo hai PNG dùng trong mục 3.1.

In [2]:
subprocess.run([sys.executable, 'scripts/chapter_03/plot_chapter3_architecture.py'], check=True)


/home/kedora/SEM5/Bigdata/taxi-data-baker-/data/benchmarks/chapter3_pipeline_architecture.png
/home/kedora/SEM5/Bigdata/taxi-data-baker-/data/benchmarks/chapter3_spark_topology.png


CompletedProcess(args=['/home/kedora/SEM5/Bigdata/taxi-data-baker-/.venv/bin/python', 'scripts/chapter_03/plot_chapter3_architecture.py'], returncode=0)

## Hình 3.3 — phân bố pickup và P95
Tính lại P95 từ Silver và yêu cầu khớp hotspot artifact đã lưu.

In [3]:
subprocess.run([sys.executable, 'scripts/chapter_03/plot_figure_3_3_pickup_p95.py'], check=True)


Silver pickup zones: 261
Nearest-rank P95: rank 248, threshold 194,522
Hotspot zones: 14 (138, 61, 132, 76, 37, 79, 181, 255, 230, 161, 7, 112, 231, 234)
Saved figure: /home/kedora/SEM5/Bigdata/taxi-data-baker-/data/benchmarks/figure_3_3_pickup_p95_hotspots.png


CompletedProcess(args=['/home/kedora/SEM5/Bigdata/taxi-data-baker-/.venv/bin/python', 'scripts/chapter_03/plot_figure_3_3_pickup_p95.py'], returncode=0)

## Hình 3.4 — quy mô raw theo tháng
Mã tính số dòng, dung lượng và vẽ biểu đồ nằm ngay trong ô dưới. Đây là thống kê input raw, không phải benchmark scaling. Mặc định notebook ghi bản riêng trong `data/benchmarks/notebook_runs/artifact_checks/` để không ghi đè artifact chính thức. Đặt biến `SCALING_RAW_PATTERN`, `SCALING_FIGURE_PATH`, `SCALING_TABLE_PATH` nếu cần thay đường dẫn.


In [ ]:
#!/usr/bin/env python3
"""Summarize raw monthly Parquet scale and create Chapter 3 Figure 3.4."""

import glob
import os
from pathlib import Path

import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# ROOT được xác định ở ô đầu tiên của notebook.
NOTEBOOK_OUTPUT = ROOT / "data/benchmarks/notebook_runs/artifact_checks"
RAW_PATTERN = os.environ.get(
    "SCALING_RAW_PATTERN", str(ROOT / "data/raw/fhvhv_tripdata_2026-0*.parquet")
)
FIGURE_PATH = Path(os.environ.get(
    "SCALING_FIGURE_PATH", NOTEBOOK_OUTPUT / "figure_3_4_data_scale.png"
))
TABLE_PATH = Path(os.environ.get(
    "SCALING_TABLE_PATH", NOTEBOOK_OUTPUT / "figure_3_4_data_scale.csv"
))


def collect_monthly_stats(pattern: str) -> pd.DataFrame:
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No raw Parquet files match: {pattern}")

    rows = []
    for file_path in files:
        path = Path(file_path)
        month = path.stem.rsplit("-", 1)[-1]
        row_count = duckdb.execute(
            "SELECT COUNT(*) FROM read_parquet(?)", [str(path)]
        ).fetchone()[0]
        size_bytes = path.stat().st_size
        rows.append({
            "month": month,
            "file": path.name,
            "rows": int(row_count),
            "size_bytes": int(size_bytes),
            "size_mb": size_bytes / (1024 * 1024),
        })
    return pd.DataFrame(rows).sort_values("month").reset_index(drop=True)


def plot_figure(stats: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax_rows = plt.subplots(figsize=(10, 5.8))
    ax_size = ax_rows.twinx()
    x = range(len(stats))

    bars = ax_rows.bar(x, stats["rows"] / 1_000_000, width=0.58,
                       color="#3478a8", alpha=0.88, label="Số bản ghi raw")
    line, = ax_size.plot(x, stats["size_mb"], color="#e07a28", marker="o",
                         linewidth=2.2, label="Dung lượng Parquet")

    ax_rows.set_xticks(list(x), [f"Tháng {month}" for month in stats["month"]])
    ax_rows.set_ylabel("Số bản ghi (triệu)")
    ax_size.set_ylabel("Dung lượng tệp Parquet (MiB)")
    ax_rows.set_title("Hình 3.4 — Quy mô dữ liệu HVFHV raw theo tháng")
    ax_rows.grid(axis="y", alpha=0.25)
    ax_rows.bar_label(bars, labels=[f"{v:.2f}" for v in stats["rows"] / 1_000_000],
                      padding=3, fontsize=8)
    ax_rows.legend([bars, line], ["Số bản ghi raw (triệu)", "Dung lượng Parquet (MiB)"],
                   loc="upper left", frameon=True)
    fig.text(0.5, 0.01,
             "Mỗi điểm là một tệp tháng riêng; đây không phải benchmark ND2/ND3 trên dữ liệu tích lũy.",
             ha="center", fontsize=9)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    stats = collect_monthly_stats(RAW_PATTERN)
    TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    stats.to_csv(TABLE_PATH, index=False, encoding="utf-8")
    plot_figure(stats, FIGURE_PATH)

    total_rows = int(stats["rows"].sum())
    total_mb = float(stats["size_mb"].sum())
    print("| Tháng | Tệp Parquet | Dung lượng | Số bản ghi thô |")
    print("| :---: | :--- | :---: | :---: |")
    for row in stats.itertuples(index=False):
        print(f"| {row.month} | {row.file} | {row.size_mb:.1f} MiB | {row.rows:,} |")
    print(f"| **Tổng** | **{len(stats)} tháng** | **{total_mb:.1f} MiB** | **{total_rows:,}** |")
    print(f"Đã lưu hình: {FIGURE_PATH}")
    print(f"Đã lưu bảng dữ liệu: {TABLE_PATH}")


main()

from IPython.display import Image, display
display(Image(filename=str(FIGURE_PATH)))


## OD composite-key preflight
Xác nhận uniqueness của OD dimension hiện tại. Trong benchmark chính thức, đối chiếu file này với OD manifest trong run metadata.

In [5]:
subprocess.run([sys.executable, 'scripts/chapter_03/audit_od_matrix_keys.py'], check=True)


                        check                                                                            value
                od_total_rows                                                                            58747
      distinct_composite_keys                                                                            58747
    null_composite_key_groups                                                                                0
     duplicate_composite_keys                                                                                0
        excess_duplicate_rows                                                                                0
                input_od_path /home/kedora/SEM5/Bigdata/taxi-data-baker-/data/silver/od_matrix_2026_01.parquet
salting_join_dimension_unique                                                                                1
Saved: /home/kedora/SEM5/Bigdata/taxi-data-baker-/data/benchmarks/od_matrix_key_audit.csv


CompletedProcess(args=['/home/kedora/SEM5/Bigdata/taxi-data-baker-/.venv/bin/python', 'scripts/chapter_03/audit_od_matrix_keys.py'], returncode=0)